# Pandas DataFrame Basics

A compact, runnable reference covering the most-used DataFrame operations: creation, inspection, indexing/selection, additions/removals, sorting, grouping, joins, and I/O.

## Setup — imports and sample DataFrame

In [2]:
import pandas as pd
import numpy as np
np.random.seed(0)
df = pd.DataFrame({
  'id': range(1,6),
  'name': ['vamshi','krishna','Tharun','Nitish','Monish'],
  'age': np.random.randint(25,38,size=5),
  'score': np.random.randint(60,101, size=5)
})
df

,id,name,age,score
0,1,vamshi,37,63
1,2,krishna,30,99
2,3,Tharun,25,69
3,4,Nitish,28,79
4,5,Monish,36,81


## Inspecting data — shape, head/tail, info, dtypes

In [3]:
print('shape:', df.shape)
print('head:', df.head())
print('tail:', df.tail())
print('info:'); df.info()
print('dtypes:', df.dtypes)


shape: (5, 4)
head:    id     name  age  score
0   1   vamshi   37     63
1   2  krishna   30     99
2   3   Tharun   25     69
3   4   Nitish   28     79
4   5   Monish   36     81
tail:    id     name  age  score
0   1   vamshi   37     63
1   2  krishna   30     99
2   3   Tharun   25     69
3   4   Nitish   28     79
4   5   Monish   36     81
info:
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      5 non-null      int64
 1   name    5 non-null      str  
 2   age     5 non-null      int64
 3   score   5 non-null      int64
dtypes: int64(3), str(1)
memory usage: 292.0 bytes
dtypes: id       int64
name       str
age      int64
score    int64
dtype: object


## Accessing columns and rows — `[]`, dot, `loc`, `iloc`, boolean masks

In [12]:
# column access
print(df['name'])
print(df.name)  # shorthand (avoid if name shadows method)
# row access by label (loc) and position (iloc)
#df.loc[0]
print('loc row 0:', df.loc[0,["name","age"]]) #label 
print('iloc row 2:', df.iloc[2,[1,2]]) #index
# boolean filtering
print('age > 30:', df[df['age'] > 30])
# select subset of columns and rows
print('subset:', df.loc[df['score']>80, ['id','name','score']])


0     vamshi
1    krishna
2     Tharun
3     Nitish
4     Monish
Name: name, dtype: str
0     vamshi
1    krishna
2     Tharun
3     Nitish
4     Monish
Name: name, dtype: str
loc row 0: name    vamshi
age         37
Name: 0, dtype: object
iloc row 2: name    Tharun
age         25
Name: 2, dtype: object
age > 30:    id    name  age  score
0   1  vamshi   37     63
4   5  Monish   36     81
subset:    id     name  score
1   2  krishna     99
4   5   Monish     81


## Adding, modifying, and removing columns

In [13]:
# add new column vectorized
df['passed'] = df['score'] >= 70
print(df)
# compute derived column with assign (chaining)
df2 = df.assign(score_pct = lambda d: (d.score - d.score.min()) / (d.score.max() - d.score.min()))
print('with score_pct:', df2)
# drop a column (returns copy)
print('drop passed:', df2.drop(columns=['passed']))


   id     name  age  score  passed
0   1   vamshi   37     63   False
1   2  krishna   30     99    True
2   3   Tharun   25     69   False
3   4   Nitish   28     79    True
4   5   Monish   36     81    True
with score_pct:    id     name  age  score  passed  score_pct
0   1   vamshi   37     63   False   0.000000
1   2  krishna   30     99    True   1.000000
2   3   Tharun   25     69   False   0.166667
3   4   Nitish   28     79    True   0.444444
4   5   Monish   36     81    True   0.500000
drop passed:    id     name  age  score  score_pct
0   1   vamshi   37     63   0.000000
1   2  krishna   30     99   1.000000
2   3   Tharun   25     69   0.166667
3   4   Nitish   28     79   0.444444
4   5   Monish   36     81   0.500000


## Sorting and ranking

In [14]:
print(df.sort_values('score', ascending=False))
print('add rank column:', df.assign(rank=df['score'].rank(ascending=False)))


   id     name  age  score  passed
1   2  krishna   30     99    True
4   5   Monish   36     81    True
3   4   Nitish   28     79    True
2   3   Tharun   25     69   False
0   1   vamshi   37     63   False
add rank column:    id     name  age  score  passed  rank
0   1   vamshi   37     63   False   5.0
1   2  krishna   30     99    True   1.0
2   3   Tharun   25     69   False   4.0
3   4   Nitish   28     79    True   3.0
4   5   Monish   36     81    True   2.0


## Grouping & aggregation — `groupby` + `agg`/`transform`

In [16]:
# small example with city and values
dfg = pd.DataFrame({'city':['Delhi','Hyderabad','Bengaluru','Chennai','Kolkata'], 'val':[10,20,30,40,50]})
print(dfg.groupby('city').agg(total=('val','sum'), mean=('val','mean')))
# transform preserves index (useful for computing per-row group features)
print('with group-mean col:', dfg.assign(group_mean=dfg.groupby('city')['val'].transform('mean')))


           total  mean
city                  
Bengaluru     30  30.0
Chennai       40  40.0
Delhi         10  10.0
Hyderabad     20  20.0
Kolkata       50  50.0
with group-mean col:         city  val  group_mean
0      Delhi   10        10.0
1  Hyderabad   20        20.0
2  Bengaluru   30        30.0
3    Chennai   40        40.0
4    Kolkata   50        50.0


## Joins and concatenation — `merge`, `join`, `concat`

In [17]:
left = pd.DataFrame({'id':[1,2,3], 'A':[10,20,30]})
right = pd.DataFrame({'id':[2,3,4], 'B':[200,300,400]})
print('merge inner:', pd.merge(left, right, on='id', how='inner'))
print('merge left:', pd.merge(left, right, on='id', how='left'))
print('concat rows:', pd.concat([left, right], ignore_index=True))


merge inner:    id   A    B
0   2  20  200
1   3  30  300
merge left:    id   A      B
0   1  10    NaN
1   2  20  200.0
2   3  30  300.0
concat rows:    id     A      B
0   1  10.0    NaN
1   2  20.0    NaN
2   3  30.0    NaN
3   2   NaN  200.0
4   3   NaN  300.0
5   4   NaN  400.0


## Missing data — isna, dropna, fillna, interpolate

In [18]:
dft = pd.DataFrame({'a':[1,None,3], 'b':[None,2,3]})
print(dft.isna())
print('dropna:', dft.dropna())
print('fillna:', dft.fillna(0))
print('interpolate:', dft.interpolate())


       a      b
0  False   True
1   True  False
2  False  False
dropna:      a    b
2  3.0  3.0
fillna:      a    b
0  1.0  0.0
1  0.0  2.0
2  3.0  3.0
interpolate:      a    b
0  1.0  NaN
1  2.0  2.0
2  3.0  3.0


In [19]:
dft = pd.DataFrame({'a':[1,None,3], 'b':[None,2,3]})
print(dft.isna())
dft.dropna()
dft
dft.interpolate()
dft

       a      b
0  False   True
1   True  False
2  False  False


,a,b
0,1.0,NaN
1,NaN,2.0
2,3.0,3.0


## Common utilities: describe, value_counts, sample, to_csv

In [21]:
print(df.describe())
print('value_counts:', df['age'].value_counts())
print('sample:', df.sample(2))
#write to CSV (example): 
df.to_csv('data/out.csv', index=False)


             id        age      score
count  5.000000   5.000000   5.000000
mean   3.000000  31.200000  78.200000
std    1.581139   5.167204  13.754999
min    1.000000  25.000000  63.000000
25%    2.000000  28.000000  69.000000
50%    3.000000  30.000000  79.000000
75%    4.000000  36.000000  81.000000
max    5.000000  37.000000  99.000000
value_counts: age
37    1
30    1
25    1
28    1
36    1
Name: count, dtype: int64
sample:    id     name  age  score  passed
3   4   Nitish   28     79    True
1   2  krishna   30     99    True


## Copy vs view gotcha and inplace note

In [22]:
a = df[['id','age']]
b = a  # both reference same object until a copy is made
b.loc[0,'age'] = 99
print('df changed via view?', df.head())
# to avoid: use copy()
c = df[['id','age']].copy()
c.loc[0,'age'] = 10
print('original unchanged when using copy:', df.head())


df changed via view?    id     name  age  score  passed
0   1   vamshi   37     63   False
1   2  krishna   30     99    True
2   3   Tharun   25     69   False
3   4   Nitish   28     79    True
4   5   Monish   36     81    True
original unchanged when using copy:    id     name  age  score  passed
0   1   vamshi   37     63   False
1   2  krishna   30     99    True
2   3   Tharun   25     69   False
3   4   Nitish   28     79    True
4   5   Monish   36     81    True


---
**Next steps:** I created `pandas_dataframe_basics_practise.ipynb` with core examples. Would you like me to run the cells to verify outputs, add more real-world examples, or include visualizations? 